# E04 — o calendário

O capítulo anterior deixou um vigia e a proposição em que ele se apoia: os dias são
**trocáveis**, e por isso o corte no k-ésimo pior assina a taxa k/(n+1). Este caderno leva esse
vigia a uma série em que a trocabilidade é falsa de forma previsível — a carga elétrica diária
de cinco praças europeias — e mede o que acontece.

**A tentativa.** Vigiar a carga diária com o mesmo instrumento, no mesmo orçamento de um alarme
por ano.

**O que se mede.**

1. o **tamanho do calendário** em cada série, em desvios-padrão do dia;
2. os alarmes da leitura crua, e em que dia da semana eles caem;
3. o preço de ler o calendário — a memória que a partição consome, e o orçamento que sobra;
4. o **controle**: a mesma medicina numa série sem calendário semanal, onde ela só custa.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E04_calendario.json, figura em .pdf e .png.

In [1]:
# <- brinque com: PAIS, JANELA, JANELAS_LONGAS, CELULAS, INICIO_FIGURA, SEMANAS_FIGURA
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import calendario, dados, graficos, volatilidade

RAIZ = Path.cwd()
PAIS = "DK_load"             # a praca cuja carga diaria este caderno vigia
JANELA = 252                 # o mesmo ano de memoria do capitulo anterior
JANELAS_LONGAS = (252, 504, 1000)
CELULAS = (("uma celula", calendario.unica), ("dia da semana", calendario.semana),
           ("dia x trimestre", calendario.semana_e_trimestre),
           ("dia x mes", calendario.semana_e_mes))
INICIO_FIGURA = "2016-05-01"
SEMANAS_FIGURA = 9
MESES = ("janeiro", "fevereiro", "marco", "abril", "maio", "junho", "julho", "agosto",
         "setembro", "outubro", "novembro", "dezembro")

tabela = pd.read_csv(dados.ARQUIVO / "opsd_carga_diaria.csv", index_col=0, parse_dates=True)
carga = tabela[PAIS].dropna()
retornos = volatilidade.retornos_log(dados.carregar_serie("sp500.csv"))
rotulo = lambda d: "%d de %s de %d" % (d.day, MESES[d.month - 1], d.year)
print("frevolab %s | %s: %d dias, de %s a %s | sp500: %d dias" % (
    frevolab.VERSAO, PAIS, len(carga), carga.index.min().date(), carga.index.max().date(),
    len(retornos)))

frevolab 0.1.0 | DK_load: 1826 dias, de 2015-01-01 a 2019-12-31 | sp500: 6718 dias


## O tamanho do calendário

In [2]:
# O calendário de cada praça, medido em desvios-padrão do dia: é o tamanho do que se pretende ler.
linhas = []
for coluna in ("DK_load", "FR_load", "ES_load", "PL_load", "AT_load"):
    s = tabela[coluna].dropna()
    p = calendario.perfil(s, calendario.semana)
    linhas.append({"serie": coluna, "dias": len(s),
                   "amplitude (sigma)": calendario.amplitude(s, calendario.semana),
                   "domingo (%)": 100 * (p[6] / s.mean() - 1),
                   "sabado (%)": 100 * (p[5] / s.mean() - 1)})
linhas.append({"serie": "sp500", "dias": len(retornos),
               "amplitude (sigma)": calendario.amplitude(retornos, calendario.semana),
               "domingo (%)": float("nan"), "sabado (%)": float("nan")})
amplitudes = {l["serie"]: l["amplitude (sigma)"] for l in linhas}
print(pd.DataFrame(linhas).round(3).to_string(index=False))

  serie  dias  amplitude (sigma)  domingo (%)  sabado (%)
DK_load  1826              1.472      -11.046      -9.793
FR_load  1826              0.780      -10.973      -6.439
ES_load  1826              1.902      -13.235      -6.474
PL_load  1826              1.991      -15.710      -5.667
AT_load  1826              1.662      -16.198      -9.022
  sp500  6718              0.042          NaN         NaN


## O vigia no orçamento de um alarme por ano

In [3]:
# O vigia do capitulo anterior, no orcamento de um alarme por ano, lido de quatro jeitos.
anos = len(carga) / 365.25
medidas = {}
for nome, celula in CELULAS:
    al = calendario.vigia_por_celula(carga, JANELA, celula)
    n_cel = len(np.unique(celula(carga.index)))
    piso = 365.25 * calendario.orcamento_por_celula(JANELA, n_cel)
    medidas[nome] = {"alarmes": int(al.sum()), "por_ano": float(al.sum() / anos),
                     "celulas": int(n_cel), "dias_por_celula": JANELA / n_cel,
                     "piso_ano": float(piso), "datas": al.index[al.to_numpy()]}
    print("%-15s %2d celulas | %5.1f dias por celula | %4d alarmes = %5.2f por ano (o piso declara %5.2f)"
          % (nome, n_cel, JANELA / n_cel, al.sum(), al.sum() / anos, piso))
cru = medidas["uma celula"]["datas"]
por_dia = [int((cru.dayofweek == i).sum()) for i in range(7)]
print()
print("os alarmes da leitura crua, por dia da semana: %s" % dict(zip(calendario.DIAS_DA_SEMANA, por_dia)))
print("em fim de semana: %.0f%%" % (100 * float(np.mean(cru.dayofweek >= 5))))

uma celula       1 celulas | 252.0 dias por celula |   21 alarmes =  4.20 por ano (o piso declara  1.44)
dia da semana    7 celulas |  36.0 dias por celula |   68 alarmes = 13.60 por ano (o piso declara  9.87)
dia x trimestre 28 celulas |   9.0 dias por celula |  210 alarmes = 42.01 por ano (o piso declara 36.52)
dia x mes       84 celulas |   3.0 dias por celula |  402 alarmes = 80.41 por ano (o piso declara 91.31)

os alarmes da leitura crua, por dia da semana: {'segunda': 0, 'terça': 0, 'quarta': 0, 'quinta': 0, 'sexta': 0, 'sábado': 8, 'domingo': 13}
em fim de semana: 100%


## O que a partição deixa comprar

In [4]:
# A particao divide a memoria: quanto de dado compra de volta o silencio.
longas = []
for janela in JANELAS_LONGAS:
    al = calendario.vigia_por_celula(carga, janela, calendario.semana)
    longas.append({"janela (dias)": janela, "dias por celula": janela / 7,
                   "alarmes por ano": float(al.sum() / anos),
                   "piso (alarmes por ano)": float(365.25 * calendario.orcamento_por_celula(janela, 7))})
print(pd.DataFrame(longas).round(2).to_string(index=False))

 janela (dias)  dias por celula  alarmes por ano  piso (alarmes por ano)
           252            36.00             13.6                    9.87
           504            72.00              3.2                    5.00
          1000           142.86              0.0                    2.54


## O controle: uma série sem calendário semanal

In [5]:
# O controle: o indice americano nao tem calendario semanal, e a mesma medicina so custa.
anos_sp = len(retornos) / 252.0
controle = {}
for nome, celula in (("uma celula", calendario.unica), ("dia da semana", calendario.semana)):
    al = calendario.vigia_por_celula(retornos, JANELA, celula)
    n_cel = len(np.unique(celula(retornos.index)))
    piso = 252 * calendario.orcamento_por_celula(JANELA, n_cel)
    controle[nome] = {"alarmes": int(al.sum()), "por_ano": float(al.sum() / anos_sp),
                      "celulas": int(n_cel), "piso_ano": float(piso)}
    print("%-15s %2d celulas | %3d alarmes = %5.2f por ano (o piso declara %5.2f)"
          % (nome, n_cel, al.sum(), al.sum() / anos_sp, piso))
print("amplitude do perfil semanal: %.2f sigma na carga, %.2f sigma no indice"
      % (amplitudes[PAIS], amplitudes["sp500"]))

uma celula       1 celulas |  33 alarmes =  1.24 por ano (o piso declara  1.00)
dia da semana    5 celulas | 136 alarmes =  5.10 por ano (o piso declara  4.90)
amplitude do perfil semanal: 1.47 sigma na carga, 0.04 sigma no indice


## As figuras

In [6]:
# Figura 1: nove semanas de carga, com os alarmes da leitura crua marcados, e o perfil semanal.
fim = pd.Timestamp(INICIO_FIGURA) + pd.Timedelta(weeks=SEMANAS_FIGURA)
trecho = carga.loc[INICIO_FIGURA:fim]
marcados = cru[(cru >= trecho.index.min()) & (cru <= trecho.index.max())]

fig, (cima, baixo) = plt.subplots(2, 1, figsize=(9.4, 6.2), gridspec_kw={"height_ratios": [1.6, 1]})
for dia in trecho.index[trecho.index.dayofweek >= 5]:
    cima.axvspan(dia, dia + pd.Timedelta(days=1), color="#d9d9d9", alpha=0.55, lw=0)
cima.plot(trecho.index, trecho.to_numpy(), color="#1f4e79", lw=1.3)
cima.plot(marcados, carga.loc[marcados].to_numpy(), "o", color="#b03a2e", ms=6,
          label="alarme da leitura crua (fim de semana em cinza)")
cima.set_ylabel("carga (MW)")
cima.legend(frameon=False, fontsize=9)
cima.grid(alpha=0.25)
baixo.bar(range(7), calendario.perfil(carga, calendario.semana).to_numpy(), color="#1f4e79")
baixo.set_xticks(range(7))
baixo.set_xticklabels(calendario.DIAS_DA_SEMANA)
baixo.set_ylabel("nivel medio (MW)")
baixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E04_calendario", 1)
plt.close(fig)
print("alarmes no trecho da figura: %d em %d dias" % (len(marcados), len(trecho)))

alarmes no trecho da figura: 2 em 64 dias


In [7]:
# Figura 2: o preco da particao, em duas vistas.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.2))
nomes = [n for n, _ in CELULAS]
x = np.arange(len(nomes))
esq.bar(x, [medidas[n]["por_ano"] for n in nomes], 0.55, color="#1f4e79", label="alarmes por ano")
esq.plot(x, [medidas[n]["piso_ano"] for n in nomes], "o--", color="#b03a2e", lw=1.4, ms=6,
         label="o que a memoria declara")
esq.set_xticks(x)
esq.set_xticklabels(["uma celula", "semana", "sem. x trim.", "sem. x mes"], fontsize=9)
esq.set_yscale("log")
esq.set_ylabel("alarmes por ano")
esq.set_title("o que a particao cobra", fontsize=10)
esq.legend(frameon=False, fontsize=8)
esq.grid(alpha=0.25, axis="y")

x2 = np.arange(len(longas))
dir_.bar(x2, [l["alarmes por ano"] for l in longas], 0.55, color="#1f4e79", label="alarmes por ano")
dir_.plot(x2, [l["piso (alarmes por ano)"] for l in longas], "o--", color="#b03a2e", lw=1.4, ms=6,
          label="o que a memoria declara")
dir_.set_xticks(x2)
dir_.set_xticklabels(["%d dias" % l["janela (dias)"] for l in longas])
dir_.set_xlabel("janela, lendo o dia da semana")
dir_.set_ylabel("alarmes por ano")
dir_.set_title("e o que a memoria compra de volta", fontsize=10)
dir_.legend(frameon=False, fontsize=8)
dir_.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E04_calendario", 2)
plt.close(fig)
print("janelas: %s | alarmes por ano: %s" % ([l["janela (dias)"] for l in longas],
                                             [round(l["alarmes por ano"], 2) for l in longas]))

janelas: [252, 504, 1000] | alarmes por ano: [13.6, 3.2, 0.0]


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Dois painéis. Em cima, nove semanas de carga entre maio e julho: a série cai em
degraus de sete em sete dias, e as faixas cinzas marcam os fins de semana — o consumo dentro
delas é sempre menor que o dos cinco dias anteriores. Os dois pontos vermelhos são alarmes da
leitura crua, e os dois caem no cinza. Embaixo, o perfil médio dos cinco anos: segunda a quinta
em torno de 3900 MW, sexta um pouco abaixo, sábado e domingo perto de 3400. O que a figura
engana: o eixo de cima começa em 3000 e não em zero, o que exagera o serrilhado da semana; e as
faixas cinzas, que são o objeto do capítulo, ocupam quase um terço da largura — quem olha
depressa vê dois terços de dias normais, quando o que a série mostra é um ciclo que se repete
toda semana.

**Figura 2.** À esquerda, quatro barras em escala logarítmica, com a linha tracejada do que a
memória declara. As três primeiras ficam acima da linha (4,5 contra 2; 13,5 contra 10; 42
contra 38) e a quarta fica abaixo (80 contra 91). À direita, o eixo é linear e a janela cresce:
13,6 acima de 9,9; 3,2 abaixo de 5,0; e a barra de mil dias não existe, porque o valor é zero.
O que o eixo engana, e é o que mais importa aqui: barra ausente se lê como dado faltando, e não
como silêncio medido; e as duas escalas diferentes fazem a queda da direita parecer maior que a
subida da esquerda, quando o que a direita mostra é o preço em dado que a esquerda cobrou em
alarme.


In [8]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
resultado = {
    "calendario_pais": PAIS,
    "calendario_serie_dias": int(len(carga)),
    "calendario_serie_inicio": rotulo(carga.index.min()),
    "calendario_serie_fim": rotulo(carga.index.max()),
    "calendario_janela": JANELA,
    "calendario_anos": float(anos),
    "calendario_amplitude_carga": amplitudes[PAIS],
    "calendario_amplitude_indice": amplitudes["sp500"],
    "calendario_amplitude_fr": amplitudes["FR_load"],
    "calendario_amplitude_es": amplitudes["ES_load"],
    "calendario_amplitude_pl": amplitudes["PL_load"],
    "calendario_amplitude_at": amplitudes["AT_load"],
    "calendario_domingo_pct": float(100 * (calendario.perfil(carga, calendario.semana)[6] / carga.mean() - 1)),
    "calendario_cru_alarmes": medidas["uma celula"]["alarmes"],
    "calendario_cru_alarmes_ano": medidas["uma celula"]["por_ano"],
    "calendario_cru_piso_ano": medidas["uma celula"]["piso_ano"],
    "calendario_cru_fim_de_semana_pct": float(100 * np.mean(cru.dayofweek >= 5)),
    "calendario_cru_sexta": por_dia[4],
    "calendario_cru_sabado": por_dia[5],
    "calendario_cru_domingo": por_dia[6],
    "calendario_semana_alarmes_ano": medidas["dia da semana"]["por_ano"],
    "calendario_semana_piso_ano": medidas["dia da semana"]["piso_ano"],
    "calendario_semana_dias_por_celula": medidas["dia da semana"]["dias_por_celula"],
    "calendario_semana_fim_de_semana_pct": float(
        100 * np.mean(medidas["dia da semana"]["datas"].dayofweek >= 5)),
    "calendario_trimestre_alarmes_ano": medidas["dia x trimestre"]["por_ano"],
    "calendario_trimestre_piso_ano": medidas["dia x trimestre"]["piso_ano"],
    "calendario_mes_alarmes_ano": medidas["dia x mes"]["por_ano"],
    "calendario_mes_piso_ano": medidas["dia x mes"]["piso_ano"],
    "calendario_indice_cru_alarmes_ano": controle["uma celula"]["por_ano"],
    "calendario_indice_semana_alarmes_ano": controle["dia da semana"]["por_ano"],
    "calendario_indice_semana_piso_ano": controle["dia da semana"]["piso_ano"],
}
for l in longas:
    extenso = {252: "um_ano", 504: "dois_anos", 1000: "quatro_anos"}[l["janela (dias)"]]
    resultado["calendario_%s_alarmes_ano" % extenso] = float(l["alarmes por ano"])
    resultado["calendario_%s_piso_ano" % extenso] = float(l["piso (alarmes por ano)"])

caminho = Path("lab/resultados/E04_calendario.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E04_calendario.json gravado | 37 grandezas
